# Pythonによるサウンドプログラミング（Psy Trance 風）

Python を用いて音を数値的に生成し、
リズム・低音・高音を組み合わせた簡単な楽曲生成を行う。

Psy Trance を参考に、以下を作成する：
- 一定間隔で鳴る Kick
- Rolling Bass
- 周期的に変化する Lead 音

これらを合成し、音楽として再生するプログラムを作成する。

## 実行環境
以下のコマンドで必要なライブラリをインストールできる。

pip install numpy sounddevice

In [6]:
# 基本パラメータ設定
import numpy as np
import sounddevice as sd

fs = 44100      # サンプリングレート
bpm = 140       # テンポ
seconds = 8     # 再生時間
t = np.linspace(0, seconds, int(fs * seconds), False)

notes = { "G3": 196.00, "B3": 246.94, "E4": 329.63, "F4": 349.23, "G4": 392.00, "B4": 493.88 } 
melody = [ "G3", "G3", "G3", "G4", "B3", "G3", "G3", "F4", "F4", "E4", "G3", "G4", "G4", "G3", "F4", "B3" ]

## Kick

Kick は低周波数の正弦波に短い減衰エンベロープをかけ、
4拍子に配置することでダンスリズムを作る。

In [7]:
beat_duration = 60 / bpm                   # 1拍の長さ = 60秒 ÷ BPM 
samples_per_beat = int(fs * beat_duration) # 1拍あたりのサンプル数を計算 

kick = np.zeros_like(t)                    # kickという配列を作成

for i in range(0, len(t), samples_per_beat):# 1拍ごとにkickを配置するためのループ
    length = min(samples_per_beat, len(t) - i) # 配列の最後で残りが1拍より短い場合、残りのサンプル数を使用
    
    freq = np.linspace(120, 50, length)    # 周波数下がる
    wave = np.sin(2 * np.pi * freq * t[:length])
    
    env = np.exp(-np.linspace(0, 1, length) * 10)  # 短い減衰
    
    kick[i:i+length] += wave * env * 1.5   # 音量調整

## Bass（Rolling Bass）

低音には **锯齿波（saw）を用い**、BPM に同期したゲート処理で  
リズミカルなベースラインを作成する。  
Kick に合わせて音量を減衰させることでサイドチェイン風効果を表現。

In [8]:
bass_freq = 49                                                     # 49Hz は低音域の G1 に近い音
bass_wave = 2 * (bass_freq * t - np.floor(0.5 + bass_freq * t))    # 鋸歯状波（saw wave）を生成
bass_env = (np.sin(2 * np.pi * bpm / 60 * 4 * t) > 0).astype(float)# ゲート処理を作成，BPM に同期して 1小節を4拍に分割
bass = bass_wave * bass_env                                        # 波形とゲートを掛け合わせることで、音の出るタイミングを制御

# Kick に合わせて音量調整（サイドチェイン）
sidechain = 1 - np.clip(kick * 8, 0, 1)
bass *= sidechain

## Lead（高音）
高音は正弦波を用い、LFO で周波数を周期的に変化させ、
揺らぎのある Psy Trance リードを表現。
16分音符単位でメロディを配置。

In [9]:
lead = np.zeros_like(t)                 # リード用の波形を初期化

step_duration = 60 / bpm / 4            # 16分音符の長さ（秒）
step_samples = int(step_duration * fs)  # サンプル数に変換

# 全体で何個の音符ステップを生成するか計算
total_steps = len(t) // step_samples    # len(t) は音声全体のサンプル数

for step in range(total_steps):
    note = melody[step % len(melody)]   # メロディをループ
    
    start = step * step_samples
    end = start + step_samples
    if end > len(t):
        break
    
    lt = np.linspace(0, step_duration, step_samples, False) # 時間軸
    
    freq = notes[note]                                      # 音の周波数
    wave = np.sin(2 * np.pi * freq * lt)                    # 正弦波生成
    env = np.exp(-lt * 10)                                  # 減衰エンベロープ
    
    lead[start:end] += wave * env                           # 波形に加算

## ミックスと再生
Kick、Bass、Lead を合成し、音割れを防ぐため最大振幅で正規化。
sounddevice を使って音を再生。

In [10]:
# 全ての波形を同じ長さに切り揃える
length = min(len(kick), len(bass), len(lead))
kick = kick[:length]
bass = bass[:length]
lead = lead[:length]

# ミックス（音量調整）
mix = kick * 2.0 + bass * 0.5 + lead * 0.3

# 正規化（音割れ防止）
mix /= np.max(np.abs(mix))

# 再生
sd.play(mix, fs)
sd.wait()